# Daemonic work extraction with ZZ-coupled spectator qubits on IBM hardware
**Seyed Navid Elyasi**

A battery qubit $B$ (centre) and its four nearest neighbours (spectators $S_1\dots S_4$) on a square-lattice IBM processor.
No two-qubit gate is applied: the battery and the spectators interact only through the static residual $ZZ$ coupling.

**Protocol (one sensing round)**
1. *Charging*: $\hat R(\theta,\phi)=e^{-i\frac\theta2(\cos\phi\,\hat\sigma_x+\sin\phi\,\hat\sigma_y)}$ on $B$; coherent charger ($\phi=0$) or stochastic charger ($\phi$ uniformly random, unknown to the controller).
2. Spectators prepared in $|+\rangle$; idle for $\tau$ (residual $ZZ$, $T_1$, $T_2$).
3. Spectators measured mid-circuit along $|\pm_\alpha\rangle=(|0\rangle\pm e^{i\alpha}|1\rangle)/\sqrt2$ (or in $Z$), giving the record $\mathbf a=(a_1,\dots,a_4)$, 16 outcomes.
4. *Extraction*: a battery-only unitary, then $\hat\sigma_z$ readout of the battery energy, $\hat H_B=\hbar\omega\,|1\rangle\langle1|$.

**Figures of merit** (same notation as Elyasi, Rossi & Genoni, *Quantum Sci. Technol.* **10**, 025017 (2025))
* unconditional ergotropy $\mathcal E_{\rm unc}=\mathcal E(\varrho^B)$, extraction unitary $\hat U$ from the passive state of $\varrho^B=\sum_{\mathbf a}p_{\mathbf a}\varrho^B_{\mathbf a}$;
* daemonic ergotropy $\overline{\mathcal E}=\sum_{\mathbf a}p_{\mathbf a}\,\mathcal E(\varrho^B_{\mathbf a})$, conditional unitaries $\hat U_{\mathbf a}$ applied with dynamic circuits;
* experimental daemonic extracted work $\overline{\mathcal W}=E(\varrho^B)-\sum_{\mathbf a}p_{\mathbf a}E(\hat U_{\mathbf a}\varrho^B_{\mathbf a}\hat U_{\mathbf a}^\dagger)$, from $\hat\sigma_z$ measurements before and after extraction.

As in the previous work, the unitaries are **precomputed** for every record from an *ideal* model ($ZZ$ only) and a *noisy* model (all parameters measured in Job 1), and applied with classically controlled operations. A third, model-free set is built from the conditional battery tomography measured in Job 2 and can be tested in Job 3.

| Job | Content | Output |
|---|---|---|
| 1 | readout confusion, spectator Ramsey with battery in $|0\rangle,|1\rangle$ ($ZZ$ phases), battery Ramsey | `calibration.json` |
| — | ideal + noisy models → unitaries | **`simple_ergotropy_unitaries.xlsx`**, **`daemonic_ergotropy_unitaries.xlsx`** |
| 2 | tomography + simple/daemonic extraction with $\hat U^{\rm ideal}$, $\hat U^{\rm noisy}$ | `daemonic_results.xlsx`, figure, tomography sheets added to both unitary files |
| 3 (optional) | extraction with the tomography unitaries | updated results |

Everything is saved in a `run_daemon_<timestamp>/` folder; every job can be retrieved later from its saved ID.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from zzbattery import single_cycle as zd

# ================================ SETTINGS ================================
BACKEND_NAME = "ibm_phoenix"
LAYOUT = [86, 76, 85, 87, 96]          # [battery, spectator 1..4]: Q86 and its four neighbours
TAUS_US = [0.0, 5.0, 10.0, 20.0]       # idle times of the protocol (all are calibrated in Job 1)
CAL_EXTRA_WAITS_US = [1.0, 2.0, 3.0]   # extra Ramsey points for the phase-rate fits

SCENARIOS = [
    # P1: coherent charger, spectators read in Z  -> the daemon learns the ZZ phase kicks (coherence)
    dict(name="P1", charger="coherent", theta=np.pi / 2, spectator_readout="Z", shots=10_000),
    # P2: stochastic (phase-random) charger, optimal spectator axes -> the daemon learns the population
    dict(name="P2", charger="stochastic", theta=np.pi / 2, n_phases=8, spectator_readout="opt", shots=10_000),
]
# spectator_readout: "Z", "X", "Y" or "opt" (axis maximising the battery 0/1 contrast, from Job 1)
# stochastic charger: shots are split equally over n_phases equally spaced charging phases (pooled)

CAL_SHOTS = 2_000                      # per Ramsey circuit
READOUT_CAL_SHOTS = 10_000             # per readout-calibration circuit
FEEDBACK = "dynamic"                   # "dynamic" (if_test on the 4-bit record) or "postselect"
T_FF_US = 0.5                          # feed-forward latency used in the noisy model
RUN_DIR = None                         # set to an existing run folder to resume
# ==========================================================================

if RUN_DIR is None:
    RUN_DIR = time.strftime("run_daemon_%Y%m%d_%H%M%S")
os.makedirs(RUN_DIR, exist_ok=True)
SPEC_NAMES = [f"Q{q}" for q in LAYOUT[1:]]
CAL_WAITS_US = sorted(set(TAUS_US) | set(CAL_EXTRA_WAITS_US))
MANIFEST = os.path.join(RUN_DIR, "manifest.json")


def save_json(name, obj):
    json.dump(obj, open(os.path.join(RUN_DIR, name), "w"), indent=1, default=float)


def load_json(name):
    return json.load(open(os.path.join(RUN_DIR, name)))


def manifest_update(**kw):
    m = json.load(open(MANIFEST)) if os.path.exists(MANIFEST) else {}
    m.update(kw)
    json.dump(m, open(MANIFEST, "w"), indent=1, default=float)
    return m


manifest_update(backend=BACKEND_NAME, layout=LAYOUT, taus_us=TAUS_US, cal_waits_us=CAL_WAITS_US,
                scenarios=SCENARIOS, feedback=FEEDBACK, t_ff_us=T_FF_US)
print("run folder:", RUN_DIR)

In [ ]:
# ---------------- connect (IBM_QUANTUM_TOKEN / IBM_QUANTUM_INSTANCE or a saved account)
service = zd.connect()
backend = service.backend(BACKEND_NAME)
dt = backend.target.dt
props = zd.backend_properties(backend, LAYOUT)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend, initial_layout=LAYOUT)
manifest_update(qubit_properties={str(k): v for k, v in props.items()}, dt=dt)


def make_sampler():
    s = SamplerV2(mode=backend)
    s.options.dynamical_decoupling.enable = False   # DD would cancel the ZZ phase used for sensing
    s.options.twirling.enable_gates = False
    s.options.twirling.enable_measure = False
    return s


def submit(circuits, tag):
    isa = [pm.run(qc) for _, qc, _ in circuits]
    job = make_sampler().run([(c, None, shots) for c, (_, _, shots) in zip(isa, circuits)])
    save_json(f"{tag}_meta.json", [m for m, _, _ in circuits])
    manifest_update(**{f"{tag}_job_id": job.job_id(), f"{tag}_submitted": time.strftime("%Y-%m-%d %H:%M:%S")})
    print(f"{tag}: {len(circuits)} circuits submitted, job id {job.job_id()}")
    return job


def fetch(tag):
    job_id = json.load(open(MANIFEST))[f"{tag}_job_id"]
    print(f"{tag}: job {job_id}, status {service.job(job_id).status()} (waiting for the result...)")
    return zd.collect(load_json(f"{tag}_meta.json"), service.job(job_id).result())


print(backend.name, "| dt =", dt, "s | battery", f"Q{LAYOUT[0]}", "| spectators", SPEC_NAMES)
pd.DataFrame(props).T.round(4)

## Job 1 — characterisation
* readout confusion matrices of the five qubits (all $|0\rangle$, all $|1\rangle$);
* spectator Ramsey with the battery in $|0\rangle$ or $|1\rangle$: phases $\psi_{j,b}(\tau)$ and contrasts $\eta_{j,b}(\tau)$ of each spectator, measured **at the protocol idle times** (the $ZZ$ coupling is $\zeta_j=-\frac{1}{2\pi}\frac{d}{d\tau}(\psi_{j,1}-\psi_{j,0})$);
* battery Ramsey with the spectators in $|0\rangle$: phase $\chi_B(\tau)$ and contrast $\eta_B(\tau)$.

In [ ]:
cal_circuits = zd.build_calibration_circuits(CAL_WAITS_US, dt, CAL_SHOTS, READOUT_CAL_SHOTS)
print(f"Job 1: {len(cal_circuits)} circuits, {sum(s for *_, s in cal_circuits):,} shots, "
      f"~{zd.estimate_qpu_seconds(cal_circuits):.0f} s of QPU time (rough estimate)")
job1 = submit(cal_circuits, "job1")

In [ ]:
rows1 = fetch("job1")
cal = zd.analyse_calibration(rows1, dt)
save_json("calibration.json", cal)

print("battery readout: P(1|0) = %.4f, P(0|1) = %.4f;  battery Ramsey rate = %.1f kHz"
      % (cal["e01"][0], cal["e10"][0], cal["omegaB"] / zd.TWO_PI * 1e3))
display(zd.calibration_table(cal, SPEC_NAMES).round(4))

t = np.array(cal["t_actual"])
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for j, n in enumerate(SPEC_NAMES):
    axs[0].plot(t, np.degrees(cal["dpsi"][j]), "o-", label=f"{n}: {cal['zeta_kHz'][j]:.1f} kHz")
    axs[1].plot(t, cal["eta"][j][0], "o-", label=f"{n}, battery 0")
    axs[1].plot(t, cal["eta"][j][1], "x--", label=f"{n}, battery 1")
axs[1].plot(t, cal["etaB"], "ks-", label="battery Ramsey")
axs[0].set(xlabel=r"$\tau$ ($\mu$s)", ylabel=r"$\psi_{j,1}-\psi_{j,0}$ (deg)", title="ZZ phase of each spectator")
axs[1].set(xlabel=r"$\tau$ ($\mu$s)", ylabel="contrast", title="Ramsey contrasts")
for a in axs:
    a.legend(fontsize=7); a.grid(alpha=.3)
fig.tight_layout(); fig.savefig(os.path.join(RUN_DIR, "calibration.png"), dpi=160)

## Extraction unitaries (ideal and noisy model) → the two Excel files
**Ideal model**: only the $ZZ$ phases $\zeta_j$ from Job 1; no decoherence, perfect readout.

**Noisy model**: the measured spectator phases/contrasts at each $\tau$, spectator $T_1$ during the idle time (partial $ZZ$ kicks), the battery Ramsey phase and contrast, battery $T_1$, $T_2$ and the $ZZ$ kicks during the spectator readout window ($T_{\rm ro}$ + feed-forward latency), and the measured readout confusion matrices.

For each scenario and $\tau$: $\hat U$ (passive state of $\varrho^B$) and $\hat U_{\mathbf a}$ for the 16 records.

In [ ]:
ALPHAS = {(sc["name"], t): zd.readout_axes(sc, t, cal) for sc in SCENARIOS for t in TAUS_US}
save_json("readout_axes.json", {f"{k[0]}|{k[1]}": v for k, v in ALPHAS.items()})

MODEL_STATES, USETS, TABLES = {}, {}, {}
for model in ("ideal", "noisy"):
    mp = zd.model_parameters(cal, props, LAYOUT, model, t_ff_us=T_FF_US)
    MODEL_STATES[model] = {(sc["name"], t): zd.conditional_states(sc, t, ALPHAS[(sc["name"], t)], mp)
                           for sc in SCENARIOS for t in TAUS_US}
    TABLES[model] = zd.unitary_tables(model, MODEL_STATES[model], SCENARIOS, TAUS_US, dt, ALPHAS, SPEC_NAMES)
    USETS[model] = TABLES[model][3]
zd.save_unitary_sets(os.path.join(RUN_DIR, "unitaries_model.json"), USETS)

SIMPLE_XLSX = os.path.join(RUN_DIR, "simple_ergotropy_unitaries.xlsx")
DAEMON_XLSX = os.path.join(RUN_DIR, "daemonic_ergotropy_unitaries.xlsx")


def write_unitary_files(tables):
    zd.write_excel(SIMPLE_XLSX, {f"{m}_model" if m != "tomography" else m: t[0] for m, t in tables.items()},
                   zd.README_SIMPLE)
    sheets = {}
    for m, t in tables.items():
        sheets[f"{m}_model" if m != "tomography" else m] = t[1]
    for m, t in tables.items():
        sheets[f"summary_{m}"] = t[2]
    zd.write_excel(DAEMON_XLSX, sheets, zd.README_DAEMONIC)
    print("written:", SIMPLE_XLSX, "and", DAEMON_XLSX)


write_unitary_files(TABLES)
pd.concat([TABLES[m][2] for m in TABLES])[["set", "scenario", "tau_us", "energy", "ergotropy_unconditional",
                                           "daemonic_ergotropy", "daemonic_gain"]].round(4)

## Job 2 — tomography and work extraction
For every scenario, $\tau$ and charging phase: conditional battery tomography ($X,Y,Z$; the $Z$ circuit also gives $E(\varrho^B)$), simple extraction with $\hat U^{\rm ideal}$, $\hat U^{\rm noisy}$, and daemonic extraction with $\hat U^{\rm ideal}_{\mathbf a}$, $\hat U^{\rm noisy}_{\mathbf a}$ (dynamic circuits). Every circuit has the same front end (charge, idle, mid-circuit spectator readout).

In [ ]:
main_circuits = zd.build_main_circuits(SCENARIOS, TAUS_US, dt, ALPHAS, USETS, feedback=FEEDBACK)
print(f"Job 2: {len(main_circuits)} circuits, {sum(s for *_, s in main_circuits):,} shots, "
      f"~{zd.estimate_qpu_seconds(main_circuits):.0f} s of QPU time (rough estimate)")
job2 = submit(main_circuits, "job2")

In [ ]:
rows2 = fetch("job2")
SUMMARY, PER_OUTCOME, TOMO_STATES = zd.analyse_main(rows2, cal, SCENARIOS, TAUS_US, MODEL_STATES, USETS, SPEC_NAMES)

# model-free unitaries from the measured conditional states -> third sheet of both Excel files
TABLES["tomography"] = zd.unitary_tables("tomography", TOMO_STATES, SCENARIOS, TAUS_US, dt, ALPHAS, SPEC_NAMES)
USETS_TOMO = {"tomography": TABLES["tomography"][3]}
zd.save_unitary_sets(os.path.join(RUN_DIR, "unitaries_tomography.json"), USETS_TOMO)
write_unitary_files(TABLES)

RESULTS_XLSX = os.path.join(RUN_DIR, "daemonic_results.xlsx")
with pd.ExcelWriter(RESULTS_XLSX, engine="openpyxl") as xw:
    SUMMARY.to_excel(xw, sheet_name="summary", index=False)
    PER_OUTCOME.to_excel(xw, sheet_name="per_outcome", index=False)
    zd.calibration_table(cal, SPEC_NAMES).to_excel(xw, sheet_name="calibration", index=False)
print("written:", RESULTS_XLSX)

cols = ["scenario", "tau_us", "E_before", "noisy_ergotropy_unconditional", "noisy_daemonic_ergotropy",
        "tomography_ergotropy_unconditional", "tomography_daemonic_ergotropy",
        "W_simple[noisy]", "W_daemonic[ideal]", "W_daemonic[noisy]", "W_daemonic_se[noisy]",
        "predicted_W_daemonic[noisy]"]
SUMMARY[[c for c in cols if c in SUMMARY]].round(4)

In [ ]:
fig = zd.plot_results(SUMMARY, SCENARIOS, ["ideal", "noisy"], os.path.join(RUN_DIR, "daemonic_work.png"))
plt.show()

## Job 3 (optional) — extraction with the tomography unitaries
Applies $\hat U^{\rm tomo}$ and $\hat U^{\rm tomo}_{\mathbf a}$ obtained from the Job 2 tomography (model-free; they absorb any mismatch of the noisy model, e.g. in the readout window). Fresh tomography circuits are included so that $E(\varrho^B)$ is measured in the same job.

In [ ]:
job3_circuits = zd.build_main_circuits(SCENARIOS, TAUS_US, dt, ALPHAS, USETS_TOMO, feedback=FEEDBACK)
print(f"Job 3: {len(job3_circuits)} circuits, {sum(s for *_, s in job3_circuits):,} shots, "
      f"~{zd.estimate_qpu_seconds(job3_circuits):.0f} s of QPU time (rough estimate)")
job3 = submit(job3_circuits, "job3")

In [ ]:
rows3 = fetch("job3")
S3, P3, _ = zd.analyse_main(rows3, cal, SCENARIOS, TAUS_US, MODEL_STATES, USETS_TOMO, SPEC_NAMES)
extra = [c for c in S3.columns if "[tomography]" in c]
SUMMARY_ALL = SUMMARY.merge(S3[["scenario", "tau_us"] + extra], on=["scenario", "tau_us"], how="left")
with pd.ExcelWriter(RESULTS_XLSX, engine="openpyxl") as xw:
    SUMMARY_ALL.to_excel(xw, sheet_name="summary", index=False)
    PER_OUTCOME.to_excel(xw, sheet_name="per_outcome", index=False)
    P3.to_excel(xw, sheet_name="per_outcome_job3", index=False)
    zd.calibration_table(cal, SPEC_NAMES).to_excel(xw, sheet_name="calibration", index=False)
fig = zd.plot_results(SUMMARY_ALL, SCENARIOS, ["ideal", "noisy", "tomography"],
                      os.path.join(RUN_DIR, "daemonic_work_with_tomography.png"))
plt.show()
SUMMARY_ALL[["scenario", "tau_us", "W_daemonic[noisy]", "W_daemonic[tomography]", "W_daemonic_se[tomography]"]].round(4)

## Resuming after a restart
Set `RUN_DIR` in the settings cell to the run folder and run the settings and connect cells, then:

In [ ]:
# --- resume: rebuild everything from the saved files (no new job is submitted)
cal = load_json("calibration.json")
ALPHAS = {(k.split("|")[0], float(k.split("|")[1])): v for k, v in load_json("readout_axes.json").items()}
USETS = zd.load_unitary_sets(os.path.join(RUN_DIR, "unitaries_model.json"))
MODEL_STATES = {}
for model in ("ideal", "noisy"):
    mp = zd.model_parameters(cal, props, LAYOUT, model, t_ff_us=T_FF_US)
    MODEL_STATES[model] = {(sc["name"], t): zd.conditional_states(sc, t, ALPHAS[(sc["name"], t)], mp)
                           for sc in SCENARIOS for t in TAUS_US}
if os.path.exists(os.path.join(RUN_DIR, "unitaries_tomography.json")):
    USETS_TOMO = zd.load_unitary_sets(os.path.join(RUN_DIR, "unitaries_tomography.json"))
print("resumed", RUN_DIR, "| jobs:", {k: v for k, v in json.load(open(MANIFEST)).items() if k.endswith("job_id")})